# Performance dos times
## Etapa de construção de tabela gold

Esta etapa consiste em tratar os dados das partidas, renomear colunas, alterar datatypes e selecionar dados relevantes para o processo.

In [0]:
from pyspark.sql.functions import *
from datetime import datetime, timedelta
from pyspark.sql.window import Window

In [0]:
df_matches = spark.read.table("apifootball.silver.matches")
df_cards = spark.read.table("apifootball.silver.cards")

In [0]:
df_home = (
    df_matches
    .select(
        "match_id",
        col("hometeam_id").alias("team_id"),
        col("hometeam_name").alias("team_name"),
        col("hometeam_score").alias("goals_for"),
        col("awayteam_score").alias("goals_against"),

        when(col("winner") == col("hometeam_name"), 1).otherwise(0).alias("wins"),
        when(col("is_draw") == 1, 1).otherwise(0).alias("draws"),
        when(
            (col("winner") != col("hometeam_name")) &
            (col("is_draw") == 0),
            1
        ).otherwise(0).alias("losses")
    )
)

In [0]:
df_away = (
    df_matches
    .select(
        "match_id",
        col("awayteam_id").alias("team_id"),
        col("awayteam_name").alias("team_name"),
        col("awayteam_score").alias("goals_for"),
        col("hometeam_score").alias("goals_against"),

        when(col("winner") == col("awayteam_name"), 1).otherwise(0).alias("wins"),
        when(col("is_draw") == 1, 1).otherwise(0).alias("draws"),
        when(
            (col("winner") != col("awayteam_name")) &
            (col("is_draw") == 0),
            1
        ).otherwise(0).alias("losses")
    )
)

In [0]:
df_team_matches = df_home.unionByName(df_away)

In [0]:
df_matches_agg = (
    df_team_matches
    .groupBy("team_id","team_name")
   .agg(
        count("*").alias("matches"),
        sum("wins").alias("wins"),
        sum("draws").alias("draws"),
        sum("losses").alias("losses"),
        sum("goals_for").alias("goals_for"),
        sum("goals_against").alias("goals_against")
    )
    .withColumn(
        "goal_difference",
        col("goals_for") - col("goals_against")
    )
    .withColumn(
        "points",
        col("wins") * 3 + col("draws")
    )
)

In [0]:
df_cards_agg = (
    df_cards
    .groupBy("team_id","team_name")
    .agg(
        sum(when(lower(col("card_type")) == "yellow card", 1).otherwise(0)).alias("yellow_cards"),
        sum(when(lower(col("card_type")) == "red card", 1).otherwise(0)).alias("red_cards")
    )
)

In [0]:
#regra para desempate de posições  
window_spec = Window.orderBy(
    desc(col("m.points")),
    desc(col("m.wins")),
    desc(col("m.goal_difference")),
    desc(col("m.goals_for")),
    asc(col("c.red_cards")),
    asc(col("c.yellow_cards"))
)

df_agg_join = (
    df_matches_agg.alias("m")
    .join(df_cards_agg.alias("c"), col("m.team_id") == col("c.team_id"), "left")
    .select(
        row_number().over(window_spec).alias("rank"),
        col("m.team_id"),
        col("m.team_name"),
        col("m.points"),
        col("m.matches"),
        col("m.wins"),
        col("m.draws"),
        col("m.losses"),
        col("m.goals_for"),
        col("m.goals_against"),
        col("m.goal_difference"),
        col("c.yellow_cards"),
        col("c.red_cards"),
    )
    .withColumn(
        "status",
        when(col("rank") <= 6, "libertadores")
        .when((col("rank") > 6) & (col("rank") <= 12), "sulamericana")
        .when(col("rank") >= 17, "relegation")
        .otherwise("-")
    )
    .withColumn(
        "datetime_processing_br",
        expr('current_timestamp() - INTERVAL 3 HOURS')
    )
).orderBy(col("m.points").desc())

In [0]:
df_agg_join.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("apifootball.gold.team_performance")

In [0]:
#spark.read.table("apifootball.gold.team_performance").display()